In [1]:
"""
End-user LPG price allocation per pixel (Nigeria, EPSG:3857)

This notebook creates a copied output from step 4.2 and adds four columns
at pixel level:
- res_cost_kg_out_walk_ref
- res_cost_kg_out_car_ref
- cost_kg_walker
- cost_kg_driver

Formulas (for each pixel):
- cost_kg_walker = reference_point_cost + best_time_walk_min / 1000
- cost_kg_driver = reference_point_cost + best_time_car_min / 1000

Reference cost lookup logic:
- first try reseller out-price from 4.4 (resell.cost_res_kg_out)
- if not found, fallback to filling out-price (filling.cost_fil_kg_out)

This matches the modeling assumption that filling points can also serve
consumers directly.

Invalid or missing assignments get sentinel value 9999.
"""

from __future__ import annotations

import sqlite3
import struct
from pathlib import Path

DATA_DIR = Path("dataset_big")

# Input from 4.2
SOURCE_PIXEL_GPKG = DATA_DIR / "huff_preferred_distributor_per_pixel.gpkg"
PIXEL_LAYER = "pixel_preference"

# Cost source from 4.4
COST_SOURCE_GPKG = DATA_DIR / "chain_with_cost.gpkg"
RESELL_LAYER = "resell"
RESELLER_ID_COL = "id_res&fil"
RESELLER_COST_COL = "cost_res_kg_out"
FILLING_LAYER = "filling"
FILLING_COST_COL = "cost_fil_kg_out"

# Output copy (fixed required name)
OUTPUT_GPKG = DATA_DIR / "end_user_price.gpkg"

# Pixel columns from 4.2
WALK_ID_COL = "best_reseller_id_walk"
WALK_TIME_COL = "best_time_walk_min"
CAR_ID_COL = "best_reseller_id_car"
CAR_TIME_COL = "best_time_car_min"

# New output columns
OUT_COST_REF_WALK = "res_cost_kg_out_walk_ref"
OUT_COST_REF_CAR = "res_cost_kg_out_car_ref"
OUT_COST_WALK = "cost_kg_walker"
OUT_COST_CAR = "cost_kg_driver"

# Sentinel and basic guards
SENTINEL = 9999.0
MAX_VALID_TIME_MIN = 7000.0


def _list_gpkg_layers(gpkg_path: Path) -> set[str]:
    with sqlite3.connect(str(gpkg_path)) as conn:
        cur = conn.cursor()
        rows = cur.execute(
            "SELECT table_name FROM gpkg_contents WHERE data_type IN ('features', 'attributes')"
        ).fetchall()
    return {name for (name,) in rows}


def _ensure_bytes(blob) -> bytes | None:
    if blob is None:
        return None
    if isinstance(blob, (bytes, bytearray)):
        return bytes(blob)
    if isinstance(blob, memoryview):
        return blob.tobytes()
    return None


def _gpkg_envelope_xy(geom_blob) -> tuple[float | None, float | None, float | None, float | None]:
    b = _ensure_bytes(geom_blob)
    if not b or len(b) < 8:
        return (None, None, None, None)

    if b[0:2] != b"GP":
        return (None, None, None, None)

    flags = b[3]
    envelope_code = (flags >> 1) & 0b111
    endian = "<" if (flags & 0b1) == 1 else ">"

    if envelope_code == 0:
        return (None, None, None, None)

    offset = 8
    env_size_by_code = {1: 32, 2: 48, 3: 48, 4: 64}
    env_size = env_size_by_code.get(envelope_code, 0)
    if env_size == 0 or len(b) < offset + env_size:
        return (None, None, None, None)

    try:
        minx, maxx, miny, maxy = struct.unpack(endian + "4d", b[offset : offset + 32])
        return (float(minx), float(maxx), float(miny), float(maxy))
    except Exception:
        return (None, None, None, None)


def _sqlite_st_isempty(geom_blob):
    b = _ensure_bytes(geom_blob)
    return 1 if (b is None or len(b) == 0) else 0


def _sqlite_st_minx(geom_blob):
    return _gpkg_envelope_xy(geom_blob)[0]


def _sqlite_st_maxx(geom_blob):
    return _gpkg_envelope_xy(geom_blob)[1]


def _sqlite_st_miny(geom_blob):
    return _gpkg_envelope_xy(geom_blob)[2]


def _sqlite_st_maxy(geom_blob):
    return _gpkg_envelope_xy(geom_blob)[3]


def _register_gpkg_sql_functions(conn: sqlite3.Connection) -> None:
    # GeoPackage update triggers may call these spatial functions.
    conn.create_function("ST_IsEmpty", 1, _sqlite_st_isempty)
    conn.create_function("ST_MinX", 1, _sqlite_st_minx)
    conn.create_function("ST_MaxX", 1, _sqlite_st_maxx)
    conn.create_function("ST_MinY", 1, _sqlite_st_miny)
    conn.create_function("ST_MaxY", 1, _sqlite_st_maxy)


def _prepare_output_copy_fixed_name(source_path: Path, output_path: Path) -> None:
    if output_path.exists():
        try:
            output_path.unlink()
        except PermissionError as exc:
            raise RuntimeError(
                f"Cannot overwrite required output '{output_path}'. Close any process using this file and rerun."
            ) from exc
    output_path.write_bytes(source_path.read_bytes())


def _effective_cost_sql(id_col_name: str) -> str:
    return f'''COALESCE(
        (
            SELECT CAST(r."{RESELLER_COST_COL}" AS REAL)
            FROM src."{RESELL_LAYER}" AS r
            WHERE CAST(r."{RESELLER_ID_COL}" AS INTEGER) = CAST("{PIXEL_LAYER}"."{id_col_name}" AS INTEGER)
            LIMIT 1
        ),
        (
            SELECT CAST(f."{FILLING_COST_COL}" AS REAL)
            FROM src."{FILLING_LAYER}" AS f
            WHERE CAST(f."{RESELLER_ID_COL}" AS INTEGER) = CAST("{PIXEL_LAYER}"."{id_col_name}" AS INTEGER)
            LIMIT 1
        )
    )'''


print("[1/6] Checking input files and required layers...")
if not SOURCE_PIXEL_GPKG.exists():
    raise FileNotFoundError(f"Missing input GPKG from 4.2: {SOURCE_PIXEL_GPKG}")
if not COST_SOURCE_GPKG.exists():
    raise FileNotFoundError(f"Missing cost source GPKG from 4.4: {COST_SOURCE_GPKG}")

pixel_layers = _list_gpkg_layers(SOURCE_PIXEL_GPKG)
if PIXEL_LAYER not in pixel_layers:
    raise KeyError(f"Layer '{PIXEL_LAYER}' not found in {SOURCE_PIXEL_GPKG}")

cost_layers = _list_gpkg_layers(COST_SOURCE_GPKG)
for lyr in [RESELL_LAYER, FILLING_LAYER]:
    if lyr not in cost_layers:
        raise KeyError(f"Layer '{lyr}' not found in {COST_SOURCE_GPKG}")

print("[2/6] Creating working copy end_user_price.gpkg...")
_prepare_output_copy_fixed_name(SOURCE_PIXEL_GPKG, OUTPUT_GPKG)

print("[3/6] Validating schema in copied output and source cost layers...")
with sqlite3.connect(str(OUTPUT_GPKG)) as conn_out:
    _register_gpkg_sql_functions(conn_out)
    cur_out = conn_out.cursor()
    pixel_cols = [c[1] for c in cur_out.execute(f'PRAGMA table_info("{PIXEL_LAYER}")').fetchall()]

for c in [WALK_ID_COL, WALK_TIME_COL, CAR_ID_COL, CAR_TIME_COL]:
    if c not in pixel_cols:
        raise KeyError(f"Missing required pixel column '{c}' in layer '{PIXEL_LAYER}'")

with sqlite3.connect(str(COST_SOURCE_GPKG)) as conn_cost:
    cur_cost = conn_cost.cursor()
    resell_cols = [c[1] for c in cur_cost.execute(f'PRAGMA table_info("{RESELL_LAYER}")').fetchall()]
    filling_cols = [c[1] for c in cur_cost.execute(f'PRAGMA table_info("{FILLING_LAYER}")').fetchall()]

for c in [RESELLER_ID_COL, RESELLER_COST_COL]:
    if c not in resell_cols:
        raise KeyError(f"Missing required reseller column '{c}' in layer '{RESELL_LAYER}'")
for c in [RESELLER_ID_COL, FILLING_COST_COL]:
    if c not in filling_cols:
        raise KeyError(f"Missing required filling column '{c}' in layer '{FILLING_LAYER}'")

print("[4/6] Adding output columns to copied GPKG (if needed)...")
with sqlite3.connect(str(OUTPUT_GPKG)) as conn_out:
    _register_gpkg_sql_functions(conn_out)
    cur = conn_out.cursor()
    cols = [c[1] for c in cur.execute(f'PRAGMA table_info("{PIXEL_LAYER}")').fetchall()]
    if OUT_COST_REF_WALK not in cols:
        cur.execute(f'ALTER TABLE "{PIXEL_LAYER}" ADD COLUMN "{OUT_COST_REF_WALK}" REAL')
    if OUT_COST_REF_CAR not in cols:
        cur.execute(f'ALTER TABLE "{PIXEL_LAYER}" ADD COLUMN "{OUT_COST_REF_CAR}" REAL')
    if OUT_COST_WALK not in cols:
        cur.execute(f'ALTER TABLE "{PIXEL_LAYER}" ADD COLUMN "{OUT_COST_WALK}" REAL')
    if OUT_COST_CAR not in cols:
        cur.execute(f'ALTER TABLE "{PIXEL_LAYER}" ADD COLUMN "{OUT_COST_CAR}" REAL')
    conn_out.commit()

print("[5/6] Computing end-user prices for walk and car...")
attach_sql = "ATTACH DATABASE ? AS src"
updated_rows = -1

walk_ref_cost_sql = _effective_cost_sql(WALK_ID_COL)
car_ref_cost_sql = _effective_cost_sql(CAR_ID_COL)

with sqlite3.connect(str(OUTPUT_GPKG)) as conn_out:
    _register_gpkg_sql_functions(conn_out)
    cur = conn_out.cursor()
    cur.execute(attach_sql, (str(COST_SOURCE_GPKG),))

    update_sql = f'''
    UPDATE "{PIXEL_LAYER}"
    SET
        "{OUT_COST_REF_WALK}" = CASE
            WHEN "{WALK_ID_COL}" > 0
            THEN {walk_ref_cost_sql}
            ELSE NULL
        END,

        "{OUT_COST_REF_CAR}" = CASE
            WHEN "{CAR_ID_COL}" > 0
            THEN {car_ref_cost_sql}
            ELSE NULL
        END,

        "{OUT_COST_WALK}" = CASE
            WHEN "{WALK_ID_COL}" > 0
             AND "{WALK_TIME_COL}" IS NOT NULL
             AND "{WALK_TIME_COL}" >= 0
             AND "{WALK_TIME_COL}" < {MAX_VALID_TIME_MIN}
             AND ({walk_ref_cost_sql}) IS NOT NULL
            THEN ({walk_ref_cost_sql}) + CAST("{WALK_TIME_COL}" AS REAL) / 1000.0
            ELSE {SENTINEL}
        END,

        "{OUT_COST_CAR}" = CASE
            WHEN "{CAR_ID_COL}" > 0
             AND "{CAR_TIME_COL}" IS NOT NULL
             AND "{CAR_TIME_COL}" >= 0
             AND "{CAR_TIME_COL}" < {MAX_VALID_TIME_MIN}
             AND ({car_ref_cost_sql}) IS NOT NULL
            THEN ({car_ref_cost_sql}) + CAST("{CAR_TIME_COL}" AS REAL) / 1000.0
            ELSE {SENTINEL}
        END
    '''

    try:
        cur.execute(update_sql)
        updated_rows = cur.rowcount
        conn_out.commit()
    finally:
        try:
            cur.execute("DETACH DATABASE src")
        except sqlite3.Error:
            pass

print("[6/6] Done.")
print(f"Output file: {OUTPUT_GPKG}")
print(f"Updated rows in layer '{PIXEL_LAYER}': {updated_rows:,}")
print(f"Columns added/updated: {OUT_COST_REF_WALK}, {OUT_COST_REF_CAR}, {OUT_COST_WALK}, {OUT_COST_CAR}")
print("Formula used:")
print("  cost_kg_walker = reference_point_cost + best_time_walk_min / 1000")
print("  cost_kg_driver = reference_point_cost + best_time_car_min / 1000")
print(f"Primary reseller source column: {RESELLER_COST_COL}")
print(f"Fallback filling source column: {FILLING_COST_COL}")
print(f"Sentinel for invalid assignments: {SENTINEL}")

[1/6] Checking input files and required layers...
[2/6] Creating working copy end_user_price.gpkg...
[3/6] Validating schema in copied output and source cost layers...
[4/6] Adding output columns to copied GPKG (if needed)...
[5/6] Computing end-user prices for walk and car...
[6/6] Done.
Output file: dataset_big\end_user_price.gpkg
Updated rows in layer 'pixel_preference': 563,482
Columns added/updated: res_cost_kg_out_walk_ref, res_cost_kg_out_car_ref, cost_kg_walker, cost_kg_driver
Formula used:
  cost_kg_walker = reference_point_cost + best_time_walk_min / 1000
  cost_kg_driver = reference_point_cost + best_time_car_min / 1000
Primary reseller source column: cost_res_kg_out
Fallback filling source column: cost_fil_kg_out
Sentinel for invalid assignments: 9999.0


In [2]:
"""
Final QA stats for 4.5 output.

Checks:
1) output file and layer exist
2) new columns were populated
3) valid vs sentinel counts for walker/driver
4) distribution stats on valid costs
5) ID-domain consistency (how many references point to resell vs filling)
"""

from __future__ import annotations

from pathlib import Path
import sqlite3

import numpy as np
import pandas as pd
import geopandas as gpd

DATA_DIR = Path("dataset_big")
OUTPUT_GPKG = DATA_DIR / "end_user_price.gpkg"
COST_SOURCE_GPKG = DATA_DIR / "chain_with_cost.gpkg"
PIXEL_LAYER = "pixel_preference"
RESELL_LAYER = "resell"
FILLING_LAYER = "filling"
ID_COL = "id_res&fil"

WALK_ID_COL = "best_reseller_id_walk"
WALK_TIME_COL = "best_time_walk_min"
CAR_ID_COL = "best_reseller_id_car"
CAR_TIME_COL = "best_time_car_min"
OUT_COST_REF_WALK = "res_cost_kg_out_walk_ref"
OUT_COST_REF_CAR = "res_cost_kg_out_car_ref"
OUT_COST_WALK = "cost_kg_walker"
OUT_COST_CAR = "cost_kg_driver"
SENTINEL = 9999.0


def _list_gpkg_layers(gpkg_path: Path) -> set[str]:
    with sqlite3.connect(str(gpkg_path)) as conn:
        cur = conn.cursor()
        rows = cur.execute(
            "SELECT table_name FROM gpkg_contents WHERE data_type IN ('features', 'attributes')"
        ).fetchall()
    return {name for (name,) in rows}


print("[1/5] Loading output and source layers...")
if not OUTPUT_GPKG.exists():
    raise FileNotFoundError(f"Output file not found: {OUTPUT_GPKG}")
if not COST_SOURCE_GPKG.exists():
    raise FileNotFoundError(f"Cost source file not found: {COST_SOURCE_GPKG}")

layers = _list_gpkg_layers(OUTPUT_GPKG)
if PIXEL_LAYER not in layers:
    raise KeyError(f"Layer '{PIXEL_LAYER}' not found in {OUTPUT_GPKG}")

for lyr in [RESELL_LAYER, FILLING_LAYER]:
    src_layers = _list_gpkg_layers(COST_SOURCE_GPKG)
    if lyr not in src_layers:
        raise KeyError(f"Layer '{lyr}' not found in {COST_SOURCE_GPKG}")

gdf = gpd.read_file(OUTPUT_GPKG, layer=PIXEL_LAYER)
if gdf.empty:
    raise RuntimeError("Output layer is empty.")

resell = gpd.read_file(COST_SOURCE_GPKG, layer=RESELL_LAYER)
filling = gpd.read_file(COST_SOURCE_GPKG, layer=FILLING_LAYER)

required = [
    WALK_ID_COL,
    WALK_TIME_COL,
    CAR_ID_COL,
    CAR_TIME_COL,
    OUT_COST_REF_WALK,
    OUT_COST_REF_CAR,
    OUT_COST_WALK,
    OUT_COST_CAR,
]
missing = [c for c in required if c not in gdf.columns]
if missing:
    raise KeyError(f"Missing required columns in output layer: {missing}")

if ID_COL not in resell.columns:
    raise KeyError(f"Missing column '{ID_COL}' in source layer '{RESELL_LAYER}'")
if ID_COL not in filling.columns:
    raise KeyError(f"Missing column '{ID_COL}' in source layer '{FILLING_LAYER}'")


def _num_array(col_name: str) -> np.ndarray:
    values = pd.to_numeric(pd.Series(gdf[col_name]), errors="coerce")
    return np.asarray(values, dtype=np.float64)


print("[2/5] Preparing numeric arrays...")
walk_id = _num_array(WALK_ID_COL)
walk_time = _num_array(WALK_TIME_COL)
car_id = _num_array(CAR_ID_COL)
car_time = _num_array(CAR_TIME_COL)
cost_ref_walk = _num_array(OUT_COST_REF_WALK)
cost_ref_car = _num_array(OUT_COST_REF_CAR)
cost_walk = _num_array(OUT_COST_WALK)
cost_car = _num_array(OUT_COST_CAR)

resell_id_set = set(pd.to_numeric(resell[ID_COL], errors="coerce").dropna().astype(np.int64).tolist())
filling_id_set = set(pd.to_numeric(filling[ID_COL], errors="coerce").dropna().astype(np.int64).tolist())

print("[3/5] QA checks and summary metrics...")
n = len(gdf)

walk_finite = np.isfinite(cost_walk)
car_finite = np.isfinite(cost_car)

walk_sentinel = walk_finite & np.isclose(cost_walk, SENTINEL)
car_sentinel = car_finite & np.isclose(cost_car, SENTINEL)

walk_valid = walk_finite & (~walk_sentinel)
car_valid = car_finite & (~car_sentinel)

walk_formula_positive = walk_valid & (cost_walk >= (walk_time / 1000.0))
car_formula_positive = car_valid & (cost_car >= (car_time / 1000.0))

walk_ref_present = (walk_id > 0) & np.isfinite(cost_ref_walk)
car_ref_present = (car_id > 0) & np.isfinite(cost_ref_car)

walk_in_resell = np.array([int(x) in resell_id_set for x in walk_id], dtype=bool)
car_in_resell = np.array([int(x) in resell_id_set for x in car_id], dtype=bool)
walk_in_filling = np.array([int(x) in filling_id_set for x in walk_id], dtype=bool)
car_in_filling = np.array([int(x) in filling_id_set for x in car_id], dtype=bool)

print("=== END USER PRICE QA ===")
print(f"Output file selected                  : {OUTPUT_GPKG}")
print(f"Rows total                            : {n:,}")
print(f"Walker finite cost                    : {int(walk_finite.sum()):,}/{n:,}")
print(f"Driver finite cost                    : {int(car_finite.sum()):,}/{n:,}")
print(f"Walker valid (cost != {SENTINEL:.0f})          : {int(walk_valid.sum()):,}/{n:,}")
print(f"Driver valid (cost != {SENTINEL:.0f})          : {int(car_valid.sum()):,}/{n:,}")
print(f"Walker sentinel ({SENTINEL:.0f}) count         : {int(walk_sentinel.sum()):,}/{n:,}")
print(f"Driver sentinel ({SENTINEL:.0f}) count         : {int(car_sentinel.sum()):,}/{n:,}")

print("\n=== INPUT LINKAGE SANITY ===")
print(f"Rows with walk reseller id > 0         : {int((walk_id > 0).sum()):,}/{n:,}")
print(f"Rows with car reseller id > 0          : {int((car_id > 0).sum()):,}/{n:,}")
print(f"Rows with finite walk time             : {int(np.isfinite(walk_time).sum()):,}/{n:,}")
print(f"Rows with finite car time              : {int(np.isfinite(car_time).sum()):,}/{n:,}")
print(f"Rows with walk ref cost present        : {int(walk_ref_present.sum()):,}/{n:,}")
print(f"Rows with car ref cost present         : {int(car_ref_present.sum()):,}/{n:,}")

print("\n=== ID-DOMAIN COVERAGE ===")
print(f"Walk IDs found in resell              : {int(((walk_id > 0) & walk_in_resell).sum()):,}/{n:,}")
print(f"Walk IDs found in filling             : {int(((walk_id > 0) & walk_in_filling).sum()):,}/{n:,}")
print(f"Car IDs found in resell               : {int(((car_id > 0) & car_in_resell).sum()):,}/{n:,}")
print(f"Car IDs found in filling              : {int(((car_id > 0) & car_in_filling).sum()):,}/{n:,}")

if walk_valid.any():
    w = cost_walk[walk_valid]
    print("\nWalker cost min/median/mean/p95/max:",
          f"{np.nanmin(w):.4f} / {np.nanmedian(w):.4f} / {np.nanmean(w):.4f} / {np.nanpercentile(w, 95):.4f} / {np.nanmax(w):.4f}")

if car_valid.any():
    c = cost_car[car_valid]
    print("Driver cost min/median/mean/p95/max:",
          f"{np.nanmin(c):.4f} / {np.nanmedian(c):.4f} / {np.nanmean(c):.4f} / {np.nanpercentile(c, 95):.4f} / {np.nanmax(c):.4f}")

print("\nFormula consistency quick check (valid rows):")
print(f"Walker rows with cost >= time/1000    : {int(walk_formula_positive.sum()):,}/{int(walk_valid.sum()):,}")
print(f"Driver rows with cost >= time/1000    : {int(car_formula_positive.sum()):,}/{int(car_valid.sum()):,}")

print("[4/5] High-level result interpretation...")
print("If many IDs are in filling, costs are now valid thanks to reseller->filling fallback.")

print("[5/5] QA completed.")
print(f"Output checked: {OUTPUT_GPKG} | layer={PIXEL_LAYER}")

[1/5] Loading output and source layers...
[2/5] Preparing numeric arrays...
[3/5] QA checks and summary metrics...
=== END USER PRICE QA ===
Output file selected                  : dataset_big\end_user_price.gpkg
Rows total                            : 563,482
Walker finite cost                    : 563,482/563,482
Driver finite cost                    : 563,482/563,482
Walker valid (cost != 9999)          : 563,387/563,482
Driver valid (cost != 9999)          : 563,387/563,482
Walker sentinel (9999) count         : 95/563,482
Driver sentinel (9999) count         : 95/563,482

=== INPUT LINKAGE SANITY ===
Rows with walk reseller id > 0         : 563,387/563,482
Rows with car reseller id > 0          : 563,387/563,482
Rows with finite walk time             : 563,482/563,482
Rows with finite car time              : 563,482/563,482
Rows with walk ref cost present        : 563,387/563,482
Rows with car ref cost present         : 563,387/563,482

=== ID-DOMAIN COVERAGE ===
Walk IDs found in